In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math as maths
import torch, torch.nn as nn
import pandas as pd
import copy
import xarray as xr
import os

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")

attempt = 1
cmap = "Oranges"

# Data loading and visualisation 

In [ ]:
ds = xr.open_dataset(f"project3_data{attempt}.grib", engine="cfgrib")
#print(ds.head)
min_longitude = ds.longitude.values[2]
max_longitude = ds.longitude.values[-1]

cropped_ds = ds.sel(longitude=slice(ds.longitude.values[2], ds.longitude.values[-1]))
#print(cropped_ds.longitude.values)
#print(cropped_ds.head)
data = cropped_ds

#print(data.v10[0])
test_v = data.v10[0]
test_u = data.u10[0]
test_plot = np.sqrt(test_v**2 + test_u**2)

test_plot.plot(cmap = cmap)
plt.title("Test visualisation, data set [0]")
plt.show()





In [ ]:
x = data.longitude.values
y = data.latitude.values[::-1]

xx, yy = np.meshgrid(x, y)
u = data.u10[0][:][::-1]
v = data.v10[0][:][::-1]

#print(xx.shape, yy.shape, u.shape, v.shape)



u_array = np.asarray(u, dtype=float)
v_array = np.asarray(v, dtype=float)
speed = np.hypot(u_array, v_array)

plt.streamplot(
    xx,
    yy,
    u_array,
    v_array,
    color=speed,
    cmap= cmap,
    density=1.8
)

plt.colorbar(label="Wind speed (m/s)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Wind Speed and Direction")
plt.show()


In [ ]:
sea_temp = xr.open_dataset(f"sea_temp.grib", engine="cfgrib")
sea_temp_cropped = sea_temp.sel(longitude=slice(min_longitude, max_longitude))
#print(sea_temp_cropped.head)
sst = np.nan_to_num(np.asarray(sea_temp_cropped.sst[:].values) - 273.15)
#print(sst[:,:])
fig, ax = plt.subplots()
sst_plot = ax.pcolormesh(sea_temp_cropped.longitude.values, sea_temp_cropped.latitude.values, 
                         sst, cmap="coolwarm", shading="nearest")
plt.colorbar(sst_plot, ax=ax, label='Sea Surface Temperature (°C)')
plt.title('Sea Surface (red) and Land (blue)')
plt.show()

In [ ]:
# Land mask: SST is NaN over land (0 after nan_to_num)
sst_lon = sea_temp_cropped.longitude.values
sst_lat = sea_temp_cropped.latitude.values
land_mask = np.isnan(sea_temp_cropped.sst.values).astype(float)

def plot_coastline(ax=None, color="black", linewidth=1.2):
    """Draw the land/sea boundary (0.5 contour of the land mask) on ax."""
    if ax is None:
        ax = plt.gca()
    ax.contour(sst_lon, sst_lat, land_mask, levels=[0.5], colors=color, linewidths=linewidth)

fig, ax = plt.subplots()
sst_plot = ax.pcolormesh(sst_lon, sst_lat, np.where(land_mask == 1, np.nan, sst), cmap="coolwarm", shading="nearest")
plot_coastline(ax)
fig.colorbar(sst_plot, label="SST (K)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.show()

In [ ]:
print(str(data.time.values[0]))


In [ ]:

x = data.longitude.values
y = data.latitude.values[::-1]


xx, yy = np.meshgrid(x, y)

rand_idx = np.random.randint(0, len(data.time.values))

def create_image(idx, step = 2, cmap = cmap):
    u = data.u10[idx][:][::-1]
    v = data.v10[idx][:][::-1]
    date = str(data.time.values[idx])
    speed = np.hypot(u, v)

    fig, ax = plt.subplots()
    
    wind_plot = ax.pcolormesh(xx, yy, speed, cmap=cmap, shading="nearest")
    plot_coastline(ax)
    fig.colorbar(wind_plot, label="Wind speed (m/s)")
    ax.quiver(xx[::step, ::step], yy[::step, ::step], u[::step, ::step],
               v[::step, ::step], color="black", scale=150)
    
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_aspect("equal")
    ax.set_title("Wind Speed and Direction with Coastline 2026 September: \n" + f"Day {date[8:10]}, Hour {date[11:13]}")
    #plt.show()
    return fig

def create_speed_plot(arr,tag = "", step = 2, cmap = cmap):
    fig, ax = plt.subplots()
    wind_plot = ax.pcolormesh(xx, yy, arr[:][::-1], cmap=cmap, shading="nearest")
    plot_coastline(ax)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_aspect("equal")
    ax.set_title(f"{tag}Wind Speed and Direction with Coastline")
    fig.colorbar(wind_plot, label="Wind speed (m/s)")
    #plt.show()
    return fig


fig = create_image(rand_idx, step=2)

In [ ]:
import io
from PIL import Image

gif_name = "Wind_velocity.gif"
frame_step = 3        # use every 6th time step (data is hourly) to keep the file small
gif_size = (480, 360) # downscale frames so the GIF renders in markdown

def fig_to_image(fig):
    """Render a matplotlib figure to a resized PIL Image."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    buf.seek(0)
    return Image.open(buf).convert("RGB").resize(gif_size, Image.LANCZOS)

if not os.path.isfile(gif_name):
    frames = []
    for i in range(0, len(data.time.values), frame_step):
        fig = create_image(i, step=2)
        frames.append(fig_to_image(fig))
        plt.close(fig)  # Close the figure to free memory

    # Share one 128-colour palette across frames to cut file size
    palette = frames[0].quantize(colors=128)
    frames = [f.quantize(palette=palette, dither=Image.Dither.NONE) for f in frames]

    frames[0].save(
        gif_name,
        save_all=True,
        append_images=frames[1:],
        duration = 200,
        loop=0,
        optimize=True
    )
    print(f"Saved {gif_name} ({len(frames)} frames, {os.path.getsize(gif_name) / 1e6:.1f} MB)")
else:
    print(f"{gif_name} already exists. Skipping GIF creation.")


In [ ]:
from IPython import display
display.Image(filename=gif_name)

# Begin VAE

Split randomly or by time?

In [ ]:
K = 16
beta = 1.0
SEED = 42

In [ ]:
# Grid is 30 (lat) x 32 (lon); add one row top and bottom to get 32 x 32.
# 'edge' repeats the boundary row, so t2m (in Kelvin) doesn't get fake 0 K rows.
data_pad = data[["u10", "v10", "t2m"]].pad(latitude=(1, 1), mode="edge")
print(data_pad.u10.shape, data_pad.v10.shape, data_pad.t2m.shape)

In [ ]:
idx = np.arange(data_pad.time.size)
train_idx = idx[idx % 7 > 1]
val_idx   = idx[idx % 7 == 1]
test_idx  = idx[idx % 7 == 0]
wind_speed = np.sqrt(data_pad.v10**2 + data_pad.u10**2)



#print(data_pad.v10.shape, data_pad.u10.shape, data_pad.t2m.shape)

train_v_mean = float(np.asarray(np.mean(data_pad.v10[train_idx])))
train_u_mean = float(np.asarray(np.mean(data_pad.u10[train_idx])))
train_speed_mean = float(np.asarray(np.mean(wind_speed[train_idx])))
train_t2m_mean = float(np.asarray(np.mean(data_pad.t2m[train_idx])))

train_v_std = float(np.asarray(np.std(data_pad.v10[train_idx])))
train_u_std = float(np.asarray(np.std(data_pad.u10[train_idx])))
train_speed_std = float(np.asarray(np.std(wind_speed[train_idx])))
train_t2m_std = float(np.asarray(np.std(data_pad.t2m[train_idx])))

X_train = np.asarray(wind_speed[train_idx])
X_val   = np.asarray(wind_speed[val_idx])
X_test  = np.asarray(wind_speed[test_idx])
 

In [ ]:
def standardize_wind(X):
    return torch.from_numpy(
        ((X - train_speed_mean) / train_speed_std).astype(np.float32)
    ).unsqueeze(1)

def unstandardize_wind(X):
    return (X * train_speed_std) + train_speed_mean

spe_tr = standardize_wind(X_train)
spe_va = standardize_wind(X_val)
spe_te = standardize_wind(X_test)

spe_tr_flat = spe_tr.reshape(spe_tr.shape[0], -1)
spe_te_flat = spe_te.reshape(spe_te.shape[0], -1)

print(f"spe_tr {spe_tr.shape}, spe_te {spe_te.shape}, flattened D = {spe_tr_flat.shape[1]}," +
      f" spe_tr_flat {spe_tr_flat.shape}, spe_te_flat {spe_te_flat.shape}")

In [ ]:
spe_tr_np = spe_tr_flat.numpy().astype(np.float64)
spe_te_np = spe_te_flat.numpy().astype(np.float64)

pca_mean = np.mean(spe_tr_np, axis=0)
x_cen = spe_tr_np - pca_mean
U, S, Vt = np.linalg.svd(x_cen, full_matrices=False)
eig_vals = S**2 / (len(x_cen) - 1)

def pca_reconstruct(X, k):
    Z = (X - pca_mean) @ Vt[:k].T
    return Z @ Vt[:k] + pca_mean, Z

def mse(a, b):
    return np.mean((a - b) ** 2)

K = min(K, Vt.shape[0])

rec_train, Z_train = pca_reconstruct(spe_tr_np, K)
rec_test, Z_test = pca_reconstruct(spe_te_np, K)
MSE_PCA = mse(spe_te_np, rec_test)
MSE_MEAN = mse(spe_te_np, np.broadcast_to(pca_mean, spe_te_np.shape))

# Use a scalar normalization scale for formatting and RMSE reporting.
# train_speed_std is the standard deviation used to standardize the data.
sigma = train_speed_std
mean_speed = train_speed_mean

var_explained = eig_vals.cumsum() / eig_vals.sum()

print(f"test MSE, mean field only : {MSE_MEAN :.5f}  "
    f"(RMSE {sigma * np.sqrt(MSE_MEAN):.2f} m/s)")
print(f"test MSE, PCA k = {K}      : {MSE_PCA:.5f}  "
      f"(RMSE {sigma * np.sqrt(MSE_PCA):.2f} m/s)")



In [ ]:
x = data_pad.longitude.values
y = data_pad.latitude.values[::-1]


xx, yy = np.meshgrid(x, y)

id = np.random.randint(0, len(X_test))


print(f"X_test[{id}] shape: {X_test[id].shape}, rec_test[{id}] shape: {rec_test[id].shape}")

create_speed_plot(X_test[id], tag="Original ")
create_speed_plot(unstandardize_wind(rec_test[id]).reshape(32, 32), step=1, tag="Reconstructed ")

diff = X_test[id] - unstandardize_wind(rec_test[id]).reshape(32, 32)
create_speed_plot(diff, tag="Difference ", cmap = "coolwarm")


x = data.longitude.values
y = data.latitude.values[::-1]
xx, yy = np.meshgrid(x, y)
create_image(test_idx[id])
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
n_components = len(eig_vals)
n_show = min(50, n_components)
ax[0].plot(
    np.arange(1, n_show + 1), 100 * var_explained[:n_show], marker="o", ms=3)
ax[0].axvline(K, color="crimson", ls="--", lw=1)
ax[0].set(xlabel="number of EOFs kept", ylabel="variance explained (%)",
          title="Wind Speed grid"); ax[0].grid(alpha=0.3)

ks = [1, 2, 4, 8, 16, 32, 64, 128]
curve = [mse(spe_te_np, pca_reconstruct(spe_te_np, k)[0]) for k in ks]
ax[1].loglog(ks, curve, marker="o")
ax[1].axvline(K, color="crimson", ls="--", lw=1)
ax[1].set(xlabel="k", ylabel="held-out MSE", title="what each extra dimension buys")
ax[1].grid(alpha=0.3, which="both")


In [ ]:
def run_training(model, batch_loss, Xtrain, Xval, epochs=20, patience = 5, lr=1e-3,
                 batch_size=32, seed=SEED, verbose=True, tag="", log_every=5 , val_beta=1.0):
    g = torch.Generator(device="cpu").manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_val, best_state = float("inf"), None
    bad_epochs = 0

    n, history = len(Xtrain), []
    for ep in range(1, epochs + 1):
        model.train()
        #train_loss = 0.0
        perm = torch.randperm(n, generator=g)
        sums, seen = {}, 0
        for i in range(0, n, batch_size):
            xb = Xtrain[perm[i:i + batch_size]].to(device)
            loss, parts = batch_loss(model, xb)

            opt.zero_grad()
            loss.backward()
            opt.step()

            count = len(xb)
            #train_loss += loss.item()*len(xb)
            sums["objective"] = sums.get("objective", 0.0) + loss.detach().item() * count
            for key, value in parts.items():
                sums[key] = sums.get(key, 0.0) + value.detach().item() * count
            seen += count
        #train_loss_avg = train_loss / n
        train = {key: value / seen for key, value in sums.items()}


        model.eval()
        with torch.no_grad():
            val_loss, val_parts = vae_validation_loss(model, Xval.to(device), beta=val_beta)

        val = {"objective": val_loss.item()}
        val.update({key: value.detach().item() for key, value in val_parts.items()})

        history.append({"epoch": ep, "train": train, "val": val})
        
        if verbose and (ep == 1 or ep % log_every == 0 or ep == epochs):
            print(
                f"{tag}epoch {ep:3d} | "
                f"train objective={train['objective']:.4f}, mse={train['mse']:.6f} | "
                f"val objective={val['objective']:.4f}, mse={val['mse']:.6f}"
            )

        if np.isfinite(val["objective"]) and val["objective"] < best_val:
            best_val = val["objective"]
            best_epoch = ep
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            if verbose:
                print(f"Early stopping at epoch {ep}; best epoch was {best_epoch}.")
            break
    if best_state is None:
        raise RuntimeError("No finite validation objective was produced.")
    return history, best_state, best_epoch 
    
def n_params(model):
    return sum(p.numel() for p in model.parameters())


def in_batches(fn, X, bs=512):
    """Apply a model or one of its methods in chunks, so a laptop is never asked
    to hold every intermediate activation for the whole split at once."""
    with torch.no_grad():
        return torch.cat([fn(X[i:i + bs]) for i in range(0, len(X), bs)])



In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, k=K, input_shape=(1, 32, 32),):
        super().__init__()
        self.enc_conv = nn.Sequential(
            nn.Conv2d(input_shape[0], input_shape[0]*16, kernel_size=3, stride=2, padding=1),  # 32x32 -> 16x16
            nn.Tanh(),
            nn.Conv2d(input_shape[0]*16, input_shape[0]*32, kernel_size=3, stride=2, padding=1),  # 16x16 -> 8x8
            nn.Tanh(),
            nn.Conv2d(input_shape[0]*32, input_shape[0]*64, kernel_size=3, stride=2, padding=1),  # 8x8 -> 4x4
            nn.Tanh(),)
        self.mu = nn.Linear(input_shape[0]*64 * 4 * 4, input_shape[0]*k)
        self.logvar = nn.Linear(input_shape[0]*64 * 4 * 4, input_shape[0]*k)
        self.from_code = nn.Linear(input_shape[0]*k, input_shape[0]*64 * 4 * 4)
        self.dec_conv = nn.Sequential(
            nn.ConvTranspose2d(input_shape[0]*64, input_shape[0]*32, kernel_size=3, stride=2, padding=1, output_padding=1),  # 4x4 -> 8x8
            nn.Tanh(),
            nn.ConvTranspose2d(input_shape[0]*32, input_shape[0]*16, kernel_size=3, stride=2, padding=1, output_padding=1),  # 8x8 -> 16x16
            nn.Tanh(),
            nn.ConvTranspose2d(input_shape[0]*16, input_shape[0], kernel_size=3, stride=2, padding=1, output_padding=1),  # 16x16 -> 32x32
            )

    def encode(self, x):
        h = self.enc_conv(x).flatten(1)
        mu = self.mu(h)
        logvar = self.logvar(h)
        return  mu, logvar

    def decode(self, z):
        h = torch.tanh(self.from_code(z)).view(-1, 64, 4, 4)
        return self.dec_conv(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return self.decode(z), mu, logvar


def make_vae_loss(beta=1.0):
    def batch_loss(model, xb):
        xhat, mu, logvar = model(xb)
        rec = (xhat - xb).pow(2).flatten(1).sum(1).mean()
        kl = 0.5 * (mu.pow(2) + logvar.exp() - 1.0 - logvar).sum(1).mean()
        return rec + beta * kl, {"rec": rec, "kl": kl, "mse": rec / D}
    return batch_loss

def vae_validation_loss(model, xb, beta=1.0):
    mu, logvar = model.encode(xb)
    xhat = model.decode(mu)  # use the posterior mean instead of sampling z

    rec = (xhat - xb).pow(2).flatten(1).sum(1).mean()
    kl = 0.5 * (
        mu.pow(2) + logvar.exp() - 1.0 - logvar
    ).sum(1).mean()

    objective = rec + beta * kl
    return objective, {"rec": rec, "kl": kl, "mse": rec / D}

        


In [ ]:
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
conv_vae = ConvVAE(k=K).to(device)
print(f"ConvVAE has {n_params(conv_vae):,} parameters")

def Dimensionality(X):
    D = 1
    for x in X.shape[1:]:
        D = D*x
    return D
D = Dimensionality(spe_tr)


print(f"Input dimensionality: {D}")
print(f"K/D = {K/D:.5f} (should be <<1 for a meaningful compression)")

In [ ]:

epochs = 1000
patience = 5
hist_vae, best_state, best_epoch = run_training(
    conv_vae, make_vae_loss(beta), spe_tr, spe_va,
    epochs=1000, patience=5, lr=1e-3, batch_size=32,
    seed=SEED, verbose=False, tag="conv_vae ", log_every=1, val_beta=beta)

conv_vae.load_state_dict(best_state)
conv_vae.eval()

best_record = hist_vae[best_epoch - 1]
print(f"Best epoch: {best_epoch}")
print(f"Best validation objective: {best_record['val']['objective']:.4f}")
print(f"Best validation reconstruction MSE: {best_record['val']['mse']:.6f}")


In [ ]:
with torch.no_grad():
    x = spe_te.to(device)
    mu, logvar = conv_vae.encode(x)
    xhat_scaled = conv_vae.decode(mu)

xhat_ms = unstandardize_wind(xhat_scaled[:, 0].cpu().numpy())
truth_ms = X_test

mse_ms2 = np.mean((xhat_ms - truth_ms) ** 2)
rmse_ms = np.sqrt(mse_ms2)

print(f"VAE test MSE:  {mse_ms2:.4f} (m/s)^2")
print(f"VAE test RMSE: {rmse_ms:.4f} m/s")
print(f"PCA test RMSE: {sigma * np.sqrt(MSE_PCA):.4f} m/s")



In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ep = [h["epoch"] for h in hist_vae]
ax[0].plot(ep, [h["train"]["rec"] for h in hist_vae], label="train")
ax[0].plot(ep, [h["val"]["rec"] for h in hist_vae], label="validation")
ax[0].set(xlabel="epoch", ylabel="reconstruction term", title="first term of the ELBO")
ax[1].plot(ep, [h["train"]["kl"] for h in hist_vae], label="train")
ax[1].plot(ep, [h["val"]["kl"] for h in hist_vae], label="validation")
ax[1].set(xlabel="epoch", ylabel="KL term", title="second term of the ELBO")
ax[2].plot(ep, [h["train"]["objective"] for h in hist_vae], label="train")
ax[2].plot(ep, [h["val"]["objective"] for h in hist_vae], label="validation")
ax[2].set(xlabel="epoch", ylabel="Loss (reconstruction + beta * KL)", title="Loss")

for a in ax:
    a.grid(alpha=0.3); a.legend()
fig.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(1, figsize=(10, 10))
ax.plot(ep, [h["train"]["objective"] for h in hist_vae], marker='o', label="train")
ax.plot(ep, [h["val"]["objective"] for h in hist_vae], marker='s', label="validation")
ax.set(xlabel="epoch", ylabel="Loss", title="Final loss (reconstruction + beta * KL) for ConvVAE")
ax.set(xlim=(ep[-10], ep[-1]+1), ylim=(min([h["train"]["objective"] for h in hist_vae[-10:]] + [h["val"]["objective"] for h in hist_vae[-10:]])*0.99,
                                      max([h["train"]["objective"] for h in hist_vae[-10:]] + [h["val"]["objective"] for h in hist_vae[-10:]]) * 1.01))
ax.set(xticks=np.arange(ax.get_xlim()[0], ax.get_xlim()[1]))
ax.axvline(best_epoch, color="crimson", ls="--", lw=2, label="best validation objective")
ax.grid(alpha=0.3); ax.legend()
plt.show()

In [ ]:
def interpolate(model, z0, z1, steps=8):
    """Decode a straight-line path between two latent vectors.

    Returns an array shaped (steps, height, width), in standardized units.
    """
    device = next(model.parameters()).device

    z0 = torch.as_tensor(z0, dtype=torch.float32, device=device).reshape(1, -1)
    z1 = torch.as_tensor(z1, dtype=torch.float32, device=device).reshape(1, -1)

    if z0.shape != z1.shape:
        raise ValueError(f"Latent shapes differ: {z0.shape} vs {z1.shape}")

    t = torch.linspace(0, 1, steps, device=device).unsqueeze(1)
    z_path = (1 - t) * z0 + t * z1

    model.eval()
    with torch.no_grad():
        decoded = model.decode(z_path)  # (steps, 1, height, width)

    return decoded[:, 0].cpu().numpy()

with torch.no_grad():
    x = spe_tr.to(device)
    mu, logvar = conv_vae.encode(x)

maps_scaled = interpolate(conv_vae, mu[0], mu[-1], steps=9)
maps_ms = unstandardize_wind(maps_scaled)

In [ ]:
create_speed_plot(maps_ms[0], xx = xx, yy=yy, step=2, tag="Interpolated step 0 ")
create_speed_plot(maps_ms[-1], xx = xx, yy=yy, step=2, tag="Interpolated step final ")
plt.show()